# LesioScan — EDA, Data Cleaning, Patient-Level Splitting
**Notebook 01 — Google Colab** | Dataset PAD-UFES-20 tetap di Google Drive, source code di GitHub.

**Alur:** setup → `scripts/eda.py` → `scripts/clean.py` → audit → `scripts/prepare.py` → validasi → output Drive.

**Penting:** notebook ini mengendalikan pipeline; logika dipusatkan di `src/lesioscan/`. Jalankan cell berurutan. Jangan menjalankan training di notebook ini.

## 1. Hubungkan Drive dan ambil source code GitHub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess, os
REPO = Path('/content/lesioscan-ml')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Liex2810/lesioscan-ml.git',str(REPO)],check=True)
else:
    print('Repo sudah ada; bila perlu perbarui dengan git pull setelah perubahan dipush.')
os.chdir(REPO)
for needed in ['scripts/eda.py','scripts/clean.py','scripts/prepare.py','src/lesioscan/eda_pipeline.py','src/lesioscan/cleaning.py','src/lesioscan/splitting.py']:
    assert (REPO/needed).is_file(), f'File tidak ditemukan: {needed}. Push paket pembaruan ke GitHub terlebih dahulu.'
print('Source code terhubung:',REPO)


## 2. Path dataset dan output — tanpa menyalin 3 GB gambar

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/LesioScan-AI')
DATASET = DRIVE_ROOT/'datasets'/'PAD-UFES-20'
METADATA = DATASET/'metadata.csv'
EDA_OUT = DRIVE_ROOT/'experiments'/'eda'
CLEAN_OUT = DRIVE_ROOT/'experiments'/'cleaning'
SPLIT_OUT = DRIVE_ROOT/'experiments'/'splits'
assert METADATA.is_file(), f'Metadata tidak ditemukan: {METADATA}'
for folder in [EDA_OUT,CLEAN_OUT,SPLIT_OUT]:folder.mkdir(parents=True,exist_ok=True)
print('Metadata:',METADATA)
print('Gambar: semua imgs_part_* dibaca langsung dari',DATASET)


## 3. EDA otomatis — tabel, grafik, dan sampel 3 gambar per kelas
Menggunakan `scripts/eda.py` → `src/lesioscan/eda_pipeline.py`. Analisis resolusi penuh hanya dilakukan jika file CSV belum ada. Jika kamu punya `image_resolution.csv` dari analisis terdahulu, file tersebut digunakan ulang.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'scripts/eda.py','--metadata',str(METADATA),'--images',str(DATASET),'--out',str(EDA_OUT)],cwd=REPO,check=True)
print('EDA selesai:',EDA_OUT)


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
classes = pd.read_csv(EDA_OUT/'tables/class_distribution.csv')
display(classes)
display(DisplayImage(filename=str(EDA_OUT/'figures/class_distribution.png')))
display(DisplayImage(filename=str(EDA_OUT/'figures/sample_images_per_class.png')))


## 4. Data Cleaning — metadata, missing/duplicate images, integritas
`clean.py` menyimpan metadata bersih, karantina, ringkasan, dan laporan detail. Untuk pemeriksaan ulang yang cepat, centang reuse hanya jika file sumber dataset **belum berubah** dan cache integritas sudah valid.

In [ ]:
# Set True HANYA jika pernah menjalankan pemeriksaan penuh di versi pipeline ini
# dan file dataset sama persis. Default False = inspeksi integritas menyeluruh.
REUSE_INTEGRITY = False
command=[sys.executable,'scripts/clean.py','--metadata',str(METADATA),'--images',str(DATASET),'--out',str(CLEAN_OUT)]
if REUSE_INTEGRITY:command.append('--reuse-integrity')
subprocess.run(command,cwd=REPO,check=True)


## 5. Audit hasil cleaning — wajib sebelum splitting

In [ ]:
import json
from IPython.display import display
with open(CLEAN_OUT/'cleaning_summary.json',encoding='utf-8') as f:
    cleaning_summary=json.load(f)
print(json.dumps(cleaning_summary,indent=2,ensure_ascii=False))
clean_data=pd.read_csv(CLEAN_OUT/'metadata_clean.csv')
quarantine=pd.read_csv(CLEAN_OUT/'quarantine.csv')
issues=pd.read_csv(CLEAN_OUT/'tables/cleaning_issues.csv')
print('Clean:',len(clean_data),'| Quarantine:',len(quarantine))
if not issues.empty:display(issues.reason.value_counts().to_frame('jumlah'))
print('Distribusi setelah cleaning:');display(clean_data.diagnostic.value_counts().to_frame('gambar'))
assert clean_data.img_id.is_unique
assert not clean_data[['img_id','patient_id','diagnostic']].isna().any().any()
assert set(clean_data.diagnostic)=={'ACK','BCC','MEL','NEV','SCC','SEK'}
# Bila ada karantina: tinjau dulu sebelum melanjutkan.
if len(quarantine):
    display(quarantine[['img_id','patient_id','diagnostic','cleaning_issue']].head(30))
    raise RuntimeError('Ada data karantina. Periksa laporan dan putuskan perlakuan sebelum splitting.')
print('✅ Audit cleaning lolos')


## 6. Patient-Level Splitting 70/15/15
Menggunakan `scripts/prepare.py` → `src/lesioscan/splitting.py`. Pencarian kandidat berbasis pasien; pembagian ini **bukan 5-fold CV**. Jika rencana akhir menggunakan CV + holdout, jangan memakai hasil ini sebagai protokol akhir tanpa revisi.

In [ ]:
subprocess.run([sys.executable,'scripts/prepare.py','--metadata',str(CLEAN_OUT/'metadata_clean.csv'),'--out',str(SPLIT_OUT),'--seed','42','--candidates','500'],cwd=REPO,check=True)
print('Split disimpan:',SPLIT_OUT/'split.csv')


## 7. Validasi hasil splitting: pasien tidak bocor, semua kelas terwakili

In [ ]:
split_df=pd.read_csv(SPLIT_OUT/'split.csv')
assert len(split_df)==len(clean_data)
assert split_df.img_id.is_unique
assert set(split_df['split'])=={'train','val','test'}
for a,b in [('train','val'),('train','test'),('val','test')]:
    pa=set(split_df.loc[split_df['split']==a,'patient_id'])
    pb=set(split_df.loc[split_df['split']==b,'patient_id'])
    assert not pa.intersection(pb),f'Patient leakage {a} vs {b}'
print('Jumlah split:');display(split_df['split'].value_counts().to_frame('gambar'))
print('Kelas:');display(pd.crosstab(split_df.diagnostic,split_df['split']))
print('Pasien unik:');display(split_df.groupby('split').patient_id.nunique().to_frame('pasien'))
print('✅ Tidak ada overlap pasien pada train/val/test')


## 8. Output untuk Notebook 02 dan kesimpulan
Output di Google Drive: `experiments/eda/`, `experiments/cleaning/metadata_clean.csv`, `experiments/splits/split.csv`. Notebook 02 membaca kedua CSV dan dataset gambar di Drive, bukan variabel yang tersimpan di runtime ini.

In [ ]:
for p in [EDA_OUT/'eda_summary.json',CLEAN_OUT/'metadata_clean.csv',CLEAN_OUT/'cleaning_summary.json',SPLIT_OUT/'split.csv',SPLIT_OUT/'split_summary.json']:
    print('✅' if p.is_file() else '❌',p)


### Catatan interpretasi untuk laporan
- Kelas PAD-UFES-20 tidak seimbang; fokus pada Macro-F1, Balanced Accuracy, dan recall per kelas.
- Missing metadata tambahan tidak otomatis membuat gambar tidak layak untuk training gambar.
- EDA seluruh dataset hanya untuk deskripsi dan audit. Keputusan tuning perlu didasarkan pada training set.
- Pembagian pasien mengurangi risiko leakage, tetapi kemiripan gambar lintas pasien tidak otomatis terdeteksi.
- Hasil penelitian ini **bukan alat diagnosis medis**.